# BirdCLEF 2026 — Baseline Ensemble Submission

Thin wrapper around `submissions/baseline_ensemble/`. All inference logic lives in the
`inference.py` module of the uploaded `baseline_ensemble_pipeline` dataset.

**Required Kaggle datasets:**
- `birdclef-2026` (competition data, mounted at `/kaggle/input/birdclef-2026`)
- `birdclef-2026-baseline-effb0-onnx` (B0 ONNX files)
- `birdclef-2026-effv2s-onnx` (SEResNeXt ONNX files — current ad-hoc name from your repo)
- `birdclef-2026-ensemble-artifacts` (xgboost_deploy.pkl + isotonic_calibrators.pkl)
- `baseline-ensemble-pipeline` (this pipeline: config.py + inference.py)
- `onnx-runtime-whl` (offline wheel for onnxruntime, used when internet is off)

Edit paths in `config.py` if your dataset names differ.

In [ ]:
# ─── Install dependencies (offline-friendly) ───
import os, sys, subprocess
try:
    import onnxruntime as ort
except ImportError:
    wheel = '/kaggle/input/datasets/brandonkhuu/onnx-runtime-whl/onnxruntime_wheel/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl'
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-deps', '-q', wheel])
    import onnxruntime as ort
print(f'onnxruntime: {ort.__version__}')

import xgboost as xgb
print(f'xgboost:     {xgb.__version__}')
import librosa
print(f'librosa:     {librosa.__version__}')

In [ ]:
# ─── Add the pipeline to sys.path ───
# After uploading submissions/baseline_ensemble/ as a Kaggle dataset, point sys.path at it.
PIPELINE_DIR = '/kaggle/input/baseline-ensemble-pipeline'

# Local fallback for testing this notebook outside Kaggle
if not os.path.exists(PIPELINE_DIR):
    candidates = [
        os.path.abspath('submissions/baseline_ensemble'),
        os.path.abspath('../submissions/baseline_ensemble'),
    ]
    for c in candidates:
        if os.path.exists(c):
            PIPELINE_DIR = c
            break

print(f'PIPELINE_DIR: {PIPELINE_DIR}')
sys.path.insert(0, PIPELINE_DIR)

import config, inference
print(f'config.ON_KAGGLE     : {config.ON_KAGGLE}')
print(f'config.TEST_DIR      : {config.TEST_DIR}')
print(f'config.OUTPUT_PATH   : {config.OUTPUT_PATH}')
print(f'config.NN_ARCHS      : {[a["name"] for a in config.NN_ARCHS]}')

In [ ]:
# ─── Run the full ensemble pipeline ───
submission = inference.main()
submission.head()

In [ ]:
# ─── Sanity check ───
import pandas as pd
df = pd.read_csv(config.OUTPUT_PATH)
print(f'Final submission: {df.shape[0]} rows × {df.shape[1]} cols')
print(f'Columns OK      : {df.columns[0] == "row_id"}')
df.head()